# Fair Code — Audit 02: AI Hiring Bias

> *Women were hired 20.9% less than equally qualified men. The algorithm wasn't told to discriminate. It learned to.*

**Dataset:** AI Fair Recruitment Dataset — `AI_Fair_Recruitment_Dataset.csv`  
**Protected attributes:** Gender, Age  
**Fair features:** Experience Years, Technical Test Score  
**Fairness metric:** Demographic Parity  
**Model:** Random Forest Classifier  

---

Pipeline:
1. Load and explore the dataset
2. Train the biased model (gender + age included)
3. Measure the fairness gap
4. Apply the 80% rule (Four-Fifths Rule)
5. Train the fair model (merit features only)
6. Compare results

## 1. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

plt.rcParams.update({
    'figure.facecolor': '#0d0f14',
    'axes.facecolor': '#131620',
    'axes.edgecolor': '#1e2130',
    'axes.labelcolor': '#b0aec0',
    'xtick.color': '#b0aec0',
    'ytick.color': '#b0aec0',
    'text.color': '#d4cfc0',
    'grid.color': '#1e2130',
    'grid.linestyle': '--',
    'font.family': 'monospace',
    'figure.dpi': 120
})

ACCENT = '#c9a84c'
DANGER = '#9b2335'
SAFE   = '#4a7c6f'
MUTED  = '#b0aec0'

print('Libraries loaded.')

## 2. Load and Explore the Dataset

In [ ]:
df = pd.read_csv('AI Fair Recruitment/AI_Fair_Recruitment_Dataset.csv')
print(f'Dataset: {df.shape[0]:,} rows, {df.shape[1]} columns')
print(f'\nColumns: {list(df.columns)}')
df.head(3)

In [ ]:
print('Gender breakdown:')
print(df['gender'].value_counts().to_string())

print(f'\nOverall hire rate: {df["hired"].mean():.1%}')
print('\nHire rate by gender:')
print((df.groupby('gender')['hired'].mean() * 100).round(2).to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

# Hire rate by gender
hire_by_gender = df.groupby('gender')['hired'].mean() * 100
axes[0].bar(hire_by_gender.index, hire_by_gender.values,
            color=[SAFE, DANGER], width=0.4)
for i, (g, v) in enumerate(hire_by_gender.items()):
    axes[0].text(i, v + 0.3, f'{v:.1f}%', ha='center', color=ACCENT, fontsize=11)
axes[0].set_title('Hire rate by gender (raw data)', color=MUTED, fontsize=10)
axes[0].set_ylabel('Hire rate (%)')
axes[0].set_ylim(0, 30)
axes[0].grid(axis='y', alpha=0.3)

# Score distribution by gender
for gender, color in [('male', MUTED), ('female', ACCENT)]:
    subset = df[df['gender'] == gender]['test_score']
    axes[1].hist(subset, bins=20, alpha=0.6, color=color, label=gender)
axes[1].set_title('Test score distribution by gender', color=MUTED, fontsize=10)
axes[1].set_xlabel('Technical test score')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()
print('Note: score distributions are nearly identical — hire rate gap is not explained by merit.')

## 3. The 80% Rule (Four-Fifths Rule)

The EEOC's legal threshold: if the selection rate for a protected group is less than 80% of the most-selected group's rate, the process is flagged as discriminatory — no proof of intent required.

This applies directly to AI hiring models.

In [ ]:
def disparate_impact_ratio(df, prediction_col, group_col, positive_label=1):
    """Four-Fifths Rule: (lowest selection rate) / (highest selection rate)."""
    rates = (
        df.assign(_sel=(df[prediction_col] == positive_label).astype(int))
          .groupby(group_col)['_sel']
          .mean()
    )
    ratio = rates.min() / rates.max()
    return {
        'selection_rates': rates.to_dict(),
        'disparate_impact_ratio': round(ratio, 3),
        'passes_four_fifths_rule': bool(ratio >= 0.80)
    }

# Check on raw dataset (before any model)
di_raw = disparate_impact_ratio(df, 'hired', 'gender')
print('Four-Fifths Rule check — raw dataset:')
for g, r in di_raw['selection_rates'].items():
    print(f'  {g}: {r:.1%}')
print(f"\nDisparate Impact Ratio: {di_raw['disparate_impact_ratio']}")
status = '✓ PASSES' if di_raw['passes_four_fifths_rule'] else '✗ FAILS'
print(f"Four-Fifths Rule:       {status}")

## 4. Train the Biased Model

Gender and age are included alongside merit signals.

In [ ]:
X_biased = pd.get_dummies(df[['gender', 'age', 'experience_years', 'test_score']])
y = df['hired']

X_train, X_test, y_train, y_test = train_test_split(
    X_biased, y, test_size=0.2, random_state=42
)

biased_model = RandomForestClassifier(n_estimators=100, random_state=42)
biased_model.fit(X_train, y_train)
biased_preds = biased_model.predict(X_test)

results_b = X_test.copy()
results_b['pred']   = biased_preds
results_b['gender'] = df.loc[X_test.index, 'gender'].values

rates_b = results_b.groupby('gender')['pred'].mean() * 100
gap_b   = rates_b['male'] - rates_b['female']

print('--- BIASED MODEL RESULTS ---')
print(f"Male Candidate Hire Rate:   {rates_b['male']:.2f}%")
print(f"Female Candidate Hire Rate: {rates_b['female']:.2f}%")
print(f"Fairness Gap:               {gap_b:.2f}%")

di_biased = disparate_impact_ratio(results_b, 'pred', 'gender')
status = '✓ PASSES' if di_biased['passes_four_fifths_rule'] else '✗ FAILS'
print(f"Four-Fifths Rule:           {status} (ratio: {di_biased['disparate_impact_ratio']})")

## 5. Train the Fair Model

Only merit-based features: `experience_years` and `test_score`.

In [ ]:
X_fair = df[['experience_years', 'test_score']]

X_train_f, X_test_f, y_train_f, y_test_f = train_test_split(
    X_fair, y, test_size=0.2, random_state=42
)

fair_model = RandomForestClassifier(n_estimators=100, random_state=42)
fair_model.fit(X_train_f, y_train_f)
fair_preds = fair_model.predict(X_test_f)

results_f = X_test_f.copy()
results_f['pred']   = fair_preds
results_f['gender'] = df.loc[X_test_f.index, 'gender'].values

rates_f = results_f.groupby('gender')['pred'].mean() * 100
gap_f   = rates_f['male'] - rates_f['female']

print('--- MITIGATED MODEL RESULTS ---')
print(f"Male Candidate Hire Rate:   {rates_f['male']:.2f}%")
print(f"Female Candidate Hire Rate: {rates_f['female']:.2f}%")
print(f"New Fairness Gap:           {gap_f:.2f}%")

di_fair = disparate_impact_ratio(results_f, 'pred', 'gender')
status = '✓ PASSES' if di_fair['passes_four_fifths_rule'] else '✗ FAILS'
print(f"Four-Fifths Rule:           {status} (ratio: {di_fair['disparate_impact_ratio']})")

## 6. Compare Results

In [ ]:
reduction = (gap_b - gap_f) / gap_b * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('AI Hiring — Biased vs Mitigated Model', color=ACCENT, fontsize=13, y=1.02)

x = ['Male', 'Female']
for ax, rates, colors, title in [
    (axes[0], rates_b, [MUTED, DANGER], f'Biased model\nGap: {gap_b:.2f}%'),
    (axes[1], rates_f, [SAFE,  SAFE  ], f'Mitigated model\nGap: {gap_f:.2f}%')
]:
    vals = [rates['male'], rates['female']]
    bars = ax.bar(x, vals, color=colors, width=0.4)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val + 0.2,
                f'{val:.2f}%', ha='center', color=ACCENT, fontsize=11)
    ax.set_ylim(0, 28)
    ax.set_ylabel('Hire rate (%)')
    ax.set_title(title, color=MUTED, fontsize=10)
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\nSummary')
print(f'-------')
print(f'Fairness gap before:        {gap_b:.2f}%')
print(f'Fairness gap after:         {gap_f:.2f}%')
print(f'Reduction:                  {reduction:.1f}%')
print(f'Four-Fifths Rule (biased):  {di_biased["disparate_impact_ratio"]} — FAILS')
print(f'Four-Fifths Rule (fair):    {di_fair["disparate_impact_ratio"]} — PASSES')

## Key Insight

The model was never told to discriminate by gender. It inferred a gender penalty from historical hiring patterns in the training data — patterns reflecting human bias, not merit. Restricting inputs to demonstrated ability (`experience_years`, `test_score`) eliminates the channel through which that bias flows and reduces the gap from 4.51% to near-zero.

The biased model also **fails the Four-Fifths Rule** (DI ratio = 0.791 < 0.80), meaning a real EEOC complaint filed against it would survive the prima facie stage.

---

*Part of the [Fair Code project](https://github.com/yakew7/Fair-Code) by [@thefaircodeproject](https://instagram.com/thefaircodeproject)*